# 05_02 - Quantile Regression: Uncertainty-Aware Forecasting
## 1. Methodology Overview

This notebook uses the real project splits and trains the quantile regression models that feed the decision engine.
The implementation lives in `src/models/quantile_models.py`, `src/models/train_model.py`, and `src/models/evaluate_model.py`.

**Source data used in this notebook:**
- `data/processed/train_features.csv`
- `data/processed/validation_features.csv`

The goal is to estimate conditional quantiles of the next-day spot price, not to invent synthetic uncertainty.

In [5]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

project_root = Path.cwd().resolve()
if not (project_root / 'src').exists():
    project_root = Path('../../').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config.constants import DATE_COLUMN, TARGET_COLUMN
from src.models.quantile_models import get_default_quantile_feature_columns

train_path = Path('../../data/processed/train_features.csv')
validation_path = Path('../../data/processed/validation_features.csv')

train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)

train_df[DATE_COLUMN] = pd.to_datetime(train_df[DATE_COLUMN])
validation_df[DATE_COLUMN] = pd.to_datetime(validation_df[DATE_COLUMN])

feature_columns = get_default_quantile_feature_columns()
available_feature_columns = [column for column in feature_columns if column in train_df.columns and column in validation_df.columns]

print(f'Train features: {train_df.shape[0]} rows x {train_df.shape[1]} columns')
print(f'Validation features: {validation_df.shape[0]} rows x {validation_df.shape[1]} columns')
print(f'Default quantile features available: {len(available_feature_columns)} / {len(feature_columns)}')
print(f'Target column: {TARGET_COLUMN}')

display(train_df[[DATE_COLUMN, TARGET_COLUMN]].head())

Train features: 1461 rows x 155 columns
Validation features: 366 rows x 155 columns
Default quantile features available: 37 / 38
Target column: Spot_Price_SPEL


,date,Spot_Price_SPEL
0,2020-01-01,35.54
1,2020-01-02,40.00
2,2020-01-03,39.51
3,2020-01-04,35.67
4,2020-01-05,38.18


## 2. Training the Quantile Suite

The project uses gradient-boosted quantile regressors to estimate the central forecast and upper-tail risk.
The model wrapper is `src.models.train_model.train_quantile_suite(...)`, which reuses the feature configuration from the repository settings.

In [3]:
from src.models.train_model import train_quantile_suite

quantile_output = train_quantile_suite(train_df=train_df, eval_df=validation_df)

display(quantile_output.summary)
print(f'Used features: {len(quantile_output.used_features)}')
trained_quantiles = ', '.join(str(result.quantile) for result in quantile_output.results)
print(f'Quantiles trained: {trained_quantiles}')

,model_name,quantile,pinball_loss,mae,rmse
0,gbr_quantile_0.5,0.50,7.977632,15.955264,20.194186
1,gbr_quantile_0.9,0.90,4.857351,22.790294,28.525883
2,gbr_quantile_0.95,0.95,3.840220,26.262822,32.755788


Used features: 37
Quantiles trained: 0.5, 0.9, 0.95


## 3. Quantile Diagnostics

The evaluation layer checks empirical coverage and interval width using `src/models/evaluate_model.py`.
This is the bridge between forecasting and the decision engine because the policy reacts to both the central forecast and the upper tail.

In [4]:
from src.models.evaluate_model import combine_quantile_predictions, evaluate_prediction_interval, evaluate_quantile_coverage

prediction_frame = combine_quantile_predictions(quantile_output.results)
coverage_q50 = evaluate_quantile_coverage(prediction_frame, 'q_0.5')
coverage_q90 = evaluate_quantile_coverage(prediction_frame, 'q_0.9')
interval_q50_q90 = evaluate_prediction_interval(prediction_frame, 'q_0.5', 'q_0.9')

display(pd.DataFrame([coverage_q50, coverage_q90]))
display(interval_q50_q90)
display(prediction_frame.head())

,quantile,empirical_coverage,coverage_error,n_obs
0,0.5,0.483680,-0.016320,337.0
1,0.9,0.703264,-0.196736,337.0


,lower_quantile_column,upper_quantile_column,empirical_coverage,average_interval_width,n_obs
0,q_0.5,q_0.9,0.219585,16.815517,337.0


,y_true,q_0.5,q_0.9,q_0.95
28,81.28,80.235959,83.679672,86.187242
29,84.26,86.721442,86.721442,87.077401
30,70.95,83.379503,84.115509,87.446345
31,65.32,71.289227,77.015751,77.015751
32,60.32,61.630357,73.029693,73.029693


## 4. Handoff

The quantile predictions produced here are the inputs consumed by the heuristic policy and tail-risk diagnostics in the next notebook.